### Homework: Pre-training, RLHF, and Parameter Efficient Fine-Tuning

#### Instructions:
- Complete the following exercises based on the lecture material.
- Submit your completed notebook with all cells executed.

---

## Part 1: Pre-training, Continued Pre-training, and Task Training

### Question 1: Understanding Pre-training and Fine-tuning
**(a)** Explain the differences between pre-training, continued pre-training, and task-specific fine-tuning.

- Pre-training uses a large, general dataset which teaches the model to predict the next token. This training step is the longest and uses the most compute resources, and the trained model may not be very useful for specific tasks.
- Continued pre-training uses a focused dataset in a specific domain to improve the model's performance in that area. All the parameters in the model are trained during continued pre-training, but less time is spent on this step compared to pre-training
- Task-specific fine-tuning may train certain parameters or add new adapter parameters. The training data is focused on a specific task. This improves the models performance in a specific area while using much less computational resources compared to pre-training

**(b)** Why might continued pre-training on domain-specific data improve model performance?

- Large LLMs are capable of in context learning and performing tasks not found in the original training data, but the accuracy of a pre-trained model will still improve when trained on more domain specific data because the model learns specific relationships and vocabulary for data in the domain.


### Question 2: Implementing Task-specific Fine-tuning
Modify the given function to fine-tune a pre-trained language model on a custom dataset.


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

dataset = load_dataset("imdb")

# demo subsets
train_small = dataset["train"].shuffle(seed=42).select(range(2000))
eval_small  = dataset["test"].shuffle(seed=42).select(range(1000))

tokenized_train = train_small.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval  = eval_small.map(tokenize_function, batched=True, remove_columns=["text"])

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_eval  = tokenized_eval.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

trainer.train()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Accuracy
50,0.417086,0.519867,0.765000
100,0.285135,0.372284,0.842000
150,0.170087,0.463318,0.850000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=0.3085914131194826, metrics={'train_runtime': 37.0938, 'train_samples_per_second': 161.752, 'train_steps_per_second': 5.095, 'total_flos': 394666583040000.0, 'train_loss': 0.3085914131194826, 'epoch': 3.0})

Modify the function to log loss and accuracy during training.

---

## Part 2: Chat Preparation and Reinforcement Learning with Human Feedback (RLHF)

### Question 3: Understanding RLHF
**(a)** What are the key components of RLHF, and why is it used in fine-tuning chat models?

- RLHF uses feedback and ranked preferences from human labeling to train value and policy models
- After the value and policy models are trained, fine tuning methods such as PPO are used to maximize the model's alignment with the trained policies.

**(b)** How does RLHF differ from traditional supervised fine-tuning?

- In RLHF the model learns what to do in certain situations rather than simply predicting the next token more accurately

### Question 4: Implementing RLHF
Modify the given script to train a simple reward model for reinforcement learning.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class RewardModel(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(768, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.fc(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the model, loss function, and optimizer
reward_model = RewardModel().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(reward_model.parameters(), lr=1e-3)

# Create dummy training data and rewards
num_samples = 2048
training_data = torch.rand(num_samples, 768)
rewards = torch.rand(num_samples, 1)

# Create the dataset and loader
dataset = TensorDataset(training_data, rewards)
loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

# Train the model across 5 epochs
reward_model.train()
epochs = 5

for epoch in range(epochs):
    total_loss = 0.0

    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Forward
        preds = reward_model(batch_X)

        # Compute loss
        loss = criterion(preds, batch_y)

        # Backprop + update
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{epochs} | avg loss: {avg_loss:.6f}")

Epoch 1/5 | avg loss: 0.149847
Epoch 2/5 | avg loss: 0.087444
Epoch 3/5 | avg loss: 0.085288
Epoch 4/5 | avg loss: 0.084846
Epoch 5/5 | avg loss: 0.082840


Modify the script to add a loss function and optimize the reward model using a batch of training data.

---

## Part 3: Parameter Efficient Fine-Tuning

### Question 5: Understanding Parameter Efficient Fine-Tuning
**(a)** What are the advantages of parameter-efficient fine-tuning techniques over full fine-tuning?

Parameter-efficient fine-tuning does not modify the existing model weights which means that it requires far less GPU memory during the fine tuning process.

**(b)** Compare LoRA and Adapters in terms of efficiency and practical use cases.

- Both LoRA and Adapters improve fine-tuning efficiency, but LoRA has better inference time effeciency and latency because new layers are not added to the model.
- Adapters are useful for cases where many specific tasks are fine tuned for the same model because a new adapter can be created for each task and swapped out as needed

### Question 6: Implementing LoRA for Efficient Fine-Tuning
Modify the given script to fine-tune a model using LoRA.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, DataCollatorForLanguageModeling, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Flatten predictions and labels for token-level accuracy
    return accuracy_metric.compute(predictions=predictions.reshape(-1), references=labels.reshape(-1))

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
# GPT2 does not have a pad token by default, so set it to eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
# Set pad_token_id for the model's configuration
base_model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    # target modules for gpt2
    target_modules=["c_attn"]
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

raw = load_dataset("wikitext", "wikitext-2-raw-v1")

def nonempty(example):
    return example["text"] is not None and example["text"].strip() != ""

raw = raw.filter(nonempty)
raw["train"] = raw["train"].select(range(2000))
raw["validation"] = raw["validation"].select(range(500))

block_size = 128

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=block_size)

tokenized = raw.map(tokenize_fn, batched=True, remove_columns=["text"])

def group_texts(examples):
    # Concatenate then split into fixed-size blocks
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    return result

lm_datasets = tokenized.map(group_texts, batched=True)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    eval_steps=50,
    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

trainer.train()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Accuracy
50,4.388483,3.973116,0.020153
100,4.364813,3.951854,0.020419


TrainOutput(global_step=105, training_loss=4.402692722138904, metrics={'train_runtime': 32.046, 'train_samples_per_second': 103.164, 'train_steps_per_second': 3.277, 'total_flos': 216706648375296.0, 'train_loss': 4.402692722138904, 'epoch': 3.0})

Modify the script to include a fine-tuning dataset and track loss during training.

---

### Submission
Submit your completed notebook with answers and executed code outputs. Ensure that all model outputs, loss trends, and training results are included in your submission.
